# Ejercicio final — Subajuste y sobreajuste en clasificación

Este notebook estudia el comportamiento de clasificadores sobre dos bases incluidas en `scikit-learn`:

- **Wine**: clasificación de tres tipos de vino.
- **Breast Cancer Wisconsin**: clasificación binaria de tumores malignos y benignos.

Se evalúan dos esquemas de validación cruzada repetida:

- **50 % entrenamiento / 50 % prueba**.
- **40 % entrenamiento / 60 % prueba**.

También se comparan modelos configurados deliberadamente para mostrar:

- **Sobreajuste**: excelente resultado en entrenamiento, pero peor generalización.
- **Subajuste**: modelo demasiado simple, con rendimiento bajo incluso en entrenamiento.
- **Ajuste intermedio**: complejidad moderada, incluyendo **KNN con $k=7$**.

> Nota conceptual: aumentar $k$ en KNN suaviza la frontera y normalmente reduce la complejidad. Por eso $k=7$ no representa “mayor orden” en el mismo sentido que aumentar el grado de un polinomio. En este ejercicio se incluye porque fue solicitado y se contrasta con $k=1$ y $k=25$.

## 1. Instalación y librerías

In [ ]:
%pip install -q scikit-learn


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 42


## 2. Cargar y explorar las bases de datos

In [ ]:
datasets = {
    "Wine": load_wine(),
    "Breast Cancer": load_breast_cancer(),
}

for name, dataset in datasets.items():
    print("=" * 65)
    print(name)
    print("=" * 65)
    print("Observaciones:", dataset.data.shape[0])
    print("Características:", dataset.data.shape[1])
    print("Clases:", list(dataset.target_names))
    print("Distribución:", np.bincount(dataset.target))
    print()


### Distribución de las clases

La validación será estratificada para conservar estas proporciones en cada repetición.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (name, dataset) in zip(axes, datasets.items()):
    counts = np.bincount(dataset.target)
    ax.bar(dataset.target_names, counts, color=["#3498db", "#e67e22", "#2ecc71"][:len(counts)])
    ax.set_title(name)
    ax.set_ylabel("Cantidad de observaciones")
    ax.tick_params(axis="x", rotation=15)
    for index, count in enumerate(counts):
        ax.text(index, count + max(counts) * 0.02, str(count), ha="center")

plt.suptitle("Distribución de clases")
plt.tight_layout()
plt.show()


## 3. Modelos deliberadamente simples y complejos

### Familia KNN

- `k=1`: alta varianza; memoriza vecinos individuales y puede sobreajustar.
- `k=7`: configuración intermedia solicitada.
- `k=25`: frontera muy suavizada; puede subajustar.

### Familia Decision Tree

- `max_depth=1`: árbol extremadamente simple; ejemplo claro de subajuste.
- `max_depth=4`: complejidad moderada.
- Sin límite de profundidad: puede memorizar entrenamiento y sobreajustar.

KNN se incluye dentro de un `Pipeline` para que la estandarización se ajuste exclusivamente con los datos de entrenamiento de cada partición, evitando fuga de información.

In [ ]:
def build_models():
    return {
        "KNN k=1 (alta varianza)": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", KNeighborsClassifier(n_neighbors=1)),
        ]),
        "KNN k=7 (solicitado)": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", KNeighborsClassifier(n_neighbors=7)),
        ]),
        "KNN k=25 (muy suave)": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", KNeighborsClassifier(n_neighbors=25)),
        ]),
        "Tree depth=1 (simple)": DecisionTreeClassifier(
            max_depth=1, random_state=SEED
        ),
        "Tree depth=4 (moderado)": DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=3, random_state=SEED
        ),
        "Tree ilimitado (complejo)": DecisionTreeClassifier(
            max_depth=None, min_samples_leaf=1, random_state=SEED
        ),
    }


models = build_models()
print("Modelos a evaluar:")
for model_name in models:
    print("-", model_name)


## 4. Validación cruzada repetida 50/50 y 40/60

`StratifiedShuffleSplit` genera diez particiones diferentes conservando la proporción de clases:

- Esquema 50/50: cada repetición entrena con la mitad de los datos.
- Esquema 40/60: cada repetición reduce aún más los datos disponibles para aprender.

En cada repetición se calculan métricas tanto de entrenamiento como de prueba. La diferencia

$$	ext{gap}=	ext{accuracy}_{train}-	ext{accuracy}_{test}$$

ayuda a detectar sobreajuste. Un gap alto indica que el modelo funciona mucho mejor con los ejemplos que ya vio.

In [ ]:
def repeated_evaluation(dataset_name, dataset, scheme_name,
                        train_size, test_size, n_splits=10):
    X, y = dataset.data, dataset.target
    splitter = StratifiedShuffleSplit(
        n_splits=n_splits,
        train_size=train_size,
        test_size=test_size,
        random_state=SEED,
    )
    rows = []

    for split_number, (train_index, test_index) in enumerate(splitter.split(X, y), 1):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        for model_name, model in build_models().items():
            start = time.perf_counter()
            model.fit(X_train, y_train)
            fit_time = time.perf_counter() - start
            train_prediction = model.predict(X_train)
            test_prediction = model.predict(X_test)

            train_accuracy = accuracy_score(y_train, train_prediction)
            test_accuracy = accuracy_score(y_test, test_prediction)
            rows.append({
                "Dataset": dataset_name,
                "Esquema": scheme_name,
                "Repetición": split_number,
                "Modelo": model_name,
                "Accuracy train": train_accuracy,
                "Accuracy test": test_accuracy,
                "Gap": train_accuracy - test_accuracy,
                "Precision macro": precision_score(
                    y_test, test_prediction, average="macro", zero_division=0
                ),
                "Recall macro": recall_score(
                    y_test, test_prediction, average="macro", zero_division=0
                ),
                "F1 macro": f1_score(
                    y_test, test_prediction, average="macro", zero_division=0
                ),
                "Tiempo ajuste (ms)": fit_time * 1000,
            })
    return rows


all_rows = []
schemes = {
    "50 train / 50 test": (0.50, 0.50),
    "40 train / 60 test": (0.40, 0.60),
}

for dataset_name, dataset in datasets.items():
    for scheme_name, (train_size, test_size) in schemes.items():
        print(f"Evaluando {dataset_name} — {scheme_name}...")
        all_rows.extend(repeated_evaluation(
            dataset_name, dataset, scheme_name, train_size, test_size
        ))

results = pd.DataFrame(all_rows)
print(f"Evaluaciones terminadas: {len(results)} filas")
display(results.head())


## 5. Resumen de métricas

Las tablas muestran el promedio de las diez repeticiones. También se incluye la desviación estándar de accuracy de prueba para medir estabilidad.

In [ ]:
summary = (
    results.groupby(["Dataset", "Esquema", "Modelo"])
    .agg(
        train_accuracy=("Accuracy train", "mean"),
        test_accuracy=("Accuracy test", "mean"),
        test_accuracy_std=("Accuracy test", "std"),
        gap=("Gap", "mean"),
        precision_macro=("Precision macro", "mean"),
        recall_macro=("Recall macro", "mean"),
        f1_macro=("F1 macro", "mean"),
        fit_time_ms=("Tiempo ajuste (ms)", "mean"),
    )
    .reset_index()
)

for dataset_name in datasets:
    for scheme_name in schemes:
        print("\n", "=" * 90)
        print(dataset_name, "—", scheme_name)
        print("=" * 90)
        table = summary[
            (summary["Dataset"] == dataset_name)
            & (summary["Esquema"] == scheme_name)
        ].drop(columns=["Dataset", "Esquema"])
        display(table.round(4))


## 6. Accuracy de entrenamiento vs. prueba

- Barras casi iguales y altas: buena generalización.
- Entrenamiento alto y prueba notablemente menor: sobreajuste.
- Ambas barras bajas: subajuste.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharey=True)

for ax, ((dataset_name, scheme_name), group) in zip(
    axes.ravel(), summary.groupby(["Dataset", "Esquema"], sort=False)
):
    positions = np.arange(len(group))
    width = 0.38
    ax.bar(positions - width/2, group["train_accuracy"], width,
           label="Train", color="#3498db")
    ax.bar(positions + width/2, group["test_accuracy"], width,
           label="Test", color="#e67e22")
    ax.set_title(f"{dataset_name}\n{scheme_name}")
    ax.set_xticks(positions)
    ax.set_xticklabels(group["Modelo"], rotation=35, ha="right", fontsize=8)
    ax.set_ylim(0.45, 1.03)
    ax.set_ylabel("Accuracy media")
    ax.legend()

plt.suptitle("Detección visual de subajuste y sobreajuste", fontsize=16)
plt.tight_layout()
plt.show()


## 7. Gap de generalización

Un gap positivo y grande es una señal directa de sobreajuste. Un gap pequeño no garantiza por sí solo un buen modelo: si train y test son bajos, existe subajuste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

for ax, (dataset_name, dataset_group) in zip(axes, summary.groupby("Dataset", sort=False)):
    pivot = dataset_group.pivot(index="Modelo", columns="Esquema", values="gap")
    pivot.plot(kind="bar", ax=ax, color=["#8e44ad", "#16a085"])
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(dataset_name)
    ax.set_ylabel("Gap medio: accuracy train - test")
    ax.tick_params(axis="x", rotation=35, labelsize=8)
    ax.legend(title="Esquema")

plt.suptitle("Gap de generalización por modelo", fontsize=16)
plt.tight_layout()
plt.show()


## 8. Variabilidad entre las diez particiones

Los boxplots muestran si el resultado depende demasiado de la partición elegida. Esto es especialmente importante en Wine, que contiene menos observaciones.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharey=True)

for ax, ((dataset_name, scheme_name), group) in zip(
    axes.ravel(), results.groupby(["Dataset", "Esquema"], sort=False)
):
    model_names = list(models)
    values = [
        group.loc[group["Modelo"] == model_name, "Accuracy test"].to_numpy()
        for model_name in model_names
    ]
    ax.boxplot(values, tick_labels=model_names, patch_artist=True)
    ax.set_title(f"{dataset_name}\n{scheme_name}")
    ax.set_ylabel("Accuracy test")
    ax.tick_params(axis="x", rotation=35, labelsize=8)

plt.suptitle("Distribución de accuracy en validación repetida", fontsize=16)
plt.tight_layout()
plt.show()


## 9. Efecto específico de aumentar k en KNN

Esta gráfica aísla los tres valores de $k$. Permite comprobar empíricamente si pasar a $k=7$ mejora la generalización y si $k=25$ comienza a suavizar demasiado el modelo.

In [ ]:
knn_results = summary[summary["Modelo"].str.startswith("KNN")].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, (dataset_name, group) in zip(axes, knn_results.groupby("Dataset", sort=False)):
    for scheme_name, scheme_group in group.groupby("Esquema", sort=False):
        k_order = [
            "KNN k=1 (alta varianza)",
            "KNN k=7 (solicitado)",
            "KNN k=25 (muy suave)",
        ]
        ordered = scheme_group.set_index("Modelo").loc[k_order]
        ax.plot([1, 7, 25], ordered["test_accuracy"], marker="o",
                linewidth=2.5, label=scheme_name)
    ax.set_title(dataset_name)
    ax.set_xlabel("Número de vecinos k")
    ax.set_ylabel("Accuracy test media")
    ax.set_xticks([1, 7, 25])
    ax.legend()

plt.suptitle("Complejidad y generalización en KNN", fontsize=15)
plt.tight_layout()
plt.show()


## 10. Evaluación detallada de una partición 40/60

Para cada dataset se selecciona el modelo con mayor F1 macro promedio. Se vuelve a entrenar sobre una partición reproducible 40/60 y se muestran el reporte por clase y la matriz de confusión.

In [ ]:
final_models = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (dataset_name, dataset) in zip(axes, datasets.items()):
    candidates = summary[
        (summary["Dataset"] == dataset_name)
        & (summary["Esquema"] == "40 train / 60 test")
    ]
    best_name = candidates.loc[candidates["f1_macro"].idxmax(), "Modelo"]

    splitter = StratifiedShuffleSplit(
        n_splits=1, train_size=0.40, test_size=0.60, random_state=SEED
    )
    train_index, test_index = next(splitter.split(dataset.data, dataset.target))
    X_train, X_test = dataset.data[train_index], dataset.data[test_index]
    y_train, y_test = dataset.target[train_index], dataset.target[test_index]

    model = build_models()[best_name]
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    final_models[dataset_name] = model

    print("\n", "=" * 70)
    print(f"{dataset_name} — mejor modelo: {best_name}")
    print("=" * 70)
    print(classification_report(
        y_test, prediction, target_names=dataset.target_names,
        digits=4, zero_division=0
    ))

    ConfusionMatrixDisplay.from_predictions(
        y_test, prediction, display_labels=dataset.target_names,
        cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(f"{dataset_name}\n{best_name}")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 11. Visualizar los extremos del árbol

Se compara la estructura subajustada de profundidad 1 con los primeros niveles del árbol ilimitado. El segundo puede continuar creando ramas hasta memorizar casos particulares.

In [ ]:
wine = datasets["Wine"]
splitter = StratifiedShuffleSplit(
    n_splits=1, train_size=0.50, test_size=0.50, random_state=SEED
)
train_index, _ = next(splitter.split(wine.data, wine.target))
X_tree, y_tree = wine.data[train_index], wine.target[train_index]

shallow_tree = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(X_tree, y_tree)
deep_tree = DecisionTreeClassifier(random_state=SEED).fit(X_tree, y_tree)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plot_tree(shallow_tree, feature_names=wine.feature_names,
          class_names=wine.target_names, filled=True, rounded=True, ax=axes[0])
axes[0].set_title("Subajuste: árbol de profundidad 1")

plot_tree(deep_tree, feature_names=wine.feature_names,
          class_names=wine.target_names, filled=True, rounded=True,
          max_depth=3, ax=axes[1])
axes[1].set_title(f"Árbol ilimitado — primeros niveles\nProfundidad real: {deep_tree.get_depth()}")
plt.tight_layout()
plt.show()


## 12. Diagnóstico automático

Se aplican reglas descriptivas sencillas:

- `gap > 0.08`: posible sobreajuste.
- `train accuracy < 0.85`: posible subajuste.
- En otro caso: ajuste razonablemente equilibrado.

Estas reglas ayudan a resumir el experimento, pero no sustituyen el análisis de métricas, dispersión y contexto.

In [ ]:
def adjustment_diagnosis(row):
    if row["gap"] > 0.08:
        return "Posible sobreajuste"
    if row["train_accuracy"] < 0.85:
        return "Posible subajuste"
    return "Ajuste equilibrado"


diagnosis = summary.copy()
diagnosis["Diagnóstico"] = diagnosis.apply(adjustment_diagnosis, axis=1)
display(diagnosis[[
    "Dataset", "Esquema", "Modelo", "train_accuracy",
    "test_accuracy", "gap", "Diagnóstico"
]].round(4))


## Conclusiones

1. El árbol sin límite y KNN con `k=1` tienden a alcanzar accuracy de entrenamiento muy alta; su gap permite verificar si esa complejidad perjudica la generalización.
2. El árbol de profundidad 1 representa subajuste cuando ni siquiera consigue separar adecuadamente los datos de entrenamiento.
3. KNN con `k=7` suele ofrecer un compromiso entre la alta varianza de `k=1` y el suavizado de `k=25`.
4. Reducir entrenamiento de 50 % a 40 % puede aumentar la variabilidad y empeorar modelos que necesitan más ejemplos.
5. Breast Cancer tiene más observaciones que Wine, por lo que normalmente presenta resultados más estables entre particiones.
6. La comparación se basa en diez particiones estratificadas, no en una única división afortunada o desfavorable.
7. Aumentar complejidad sin validación no garantiza mejor rendimiento: debe observarse simultáneamente train, test, gap y variabilidad.